In [1]:
from experiments.bench import harness, plots
import pandas as pd
result = harness.load_result("exp2_pipeline")
print(result["environment"])

{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'machine': 'arm64', 'cpu_count': 10, 'cbc': 'Version: 2.10.3', 'packages': {'numpy': '2.5.1', 'scipy': '1.18.0', 'pulp': '3.3.2', 'neo4j': '6.2.0', 'tiktoken': '0.13.0', 'openai': '2.52.0', 'pandas': '3.0.5'}}


In [2]:
summary = pd.DataFrame([{
    "model": result["data"]["model"],
    "full_llm_tokens": result["data"]["token_counts"]["full_llm"],
    "hybrid_tokens": result["data"]["token_counts"]["hybrid"],
    "full_llm_cost_usd": result["data"]["cost_usd"]["full_llm"],
    "hybrid_cost_usd": result["data"]["cost_usd"]["hybrid"],
    "savings_pct": result["data"]["savings_pct"],
}]).round(4)
display(summary)
display(pd.DataFrame(result["data"]["sensitivity"]).round(3))

,model,full_llm_tokens,hybrid_tokens,full_llm_cost_usd,hybrid_cost_usd,savings_pct
0,gpt-5-nano,87896,60292,0.2075,0.1423,31.4053


,freetext_multiplier,full_llm_tokens,hybrid_tokens,savings_pct
0,0.25,50305,22701,54.873
1,0.50,62926,35322,43.867
2,1.00,87896,60292,31.405
3,2.00,138007,110403,20.002
4,4.00,238229,210625,11.587


In [3]:
plots.exp2_savings(result)   # figures/exp2_savings.png 생성

PosixPath('/Users/honey/Dev/teamweaver/experiments/figures/exp2_savings.png')

## 관찰 요약

- 기준 페이로드(자유서술 배수 1.0×)에서 절감률은 **31.4%**다 — 최초 가설이었던 80%대에는 크게 못 미친다.
- 민감도 곡선(`data.sensitivity`)은 자유서술 분량 배수 0.25×→4×에 걸쳐 절감률이 54.9%→11.6%로 단조 감소한다. 즉 절감률은 고정 상수가 아니라 정형/비정형 리뷰 내용 비중에 강하게 의존하는 값이다 — 비정형 서술이 많은 조직일수록 Hybrid 파이프라인의 이점이 줄어든다.
- 정확도 축(`data.accuracy`)은 이번 실행이 `live=False`(오프라인 토큰 카운팅만) 이라 **미측정**이다. 라이브로 돌리더라도 구조적 문제가 남는다 — 이번 Full-LLM 페이로드는 정답 항목 선택을 입력 필드로 그대로 전달하므로, 출력을 그 입력과 비교하면 동어반복적으로 높은 일치율이 나온다(Task 7 리포트 기록). 정직한 재추출 오류율을 재려면 항목을 가린 별도 페이로드가 필요하다.
- 지연(latency) 축(`data.latency`)도 이번 실행에서는 직접 측정하지 않았다. 참고치로 Task 6 체크포인트 실측(리뷰당 gen+parse 합산 ~24-28초)을 인용했으나, 이는 다른 프롬프트 형태로 측정된 하한(lower-bound) 성격의 수치이지 이번 Full-LLM/Hybrid 페이로드로 직접 측정된 값이 아니다.
